# 选修实验 - 多分类


## 1.1 目标
在本实验中，你将探索一个使用神经网络进行多分类的示例。
<figure>
 <img src="./images/C2_W2_mclass_header.png"   style="width500px;height:200px;">
</figure>


## 1.2 工具
你将使用一些绘图例程。它们存放在本目录下的 `lab_utils_multiclass_TF.py` 中。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
np.set_printoptions(precision=2)
from lab_utils_multiclass_TF import *
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

# 2.0 多分类
神经网络常用于数据分类。例如神经网络可以：
- 接收照片，并将照片中的主体分类为 {狗、猫、马、其他}
- 接收句子，并划分其成分的“词性”：{名词、动词、形容词等}  

这类网络的最后一层会有多个单元。每个输出对应一个类别。当输入样本送入网络后，输出值最大的那个即为预测的类别。如果将输出送入 softmax 函数，softmax 的输出将给出输入属于每个类别的概率。

在本实验中，你将看到在 Tensorflow 中构建多分类网络的示例。随后我们会看一看神经网络是如何做出预测的。

让我们先创建一个四分类数据集。

## 2.1 准备并可视化数据
我们将使用 Scikit-Learn 的 `make_blobs` 函数生成一个包含 4 个类别的训练数据集，如下图所示。

In [2]:
# 生成用于分类的 4 分类数据集
classes = 4
m = 100
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
std = 1.0
X_train, y_train = make_blobs(n_samples=m, centers=centers, cluster_std=std,random_state=30)

In [3]:
plt_mc(X_train,y_train,classes, centers, std=std)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

每个点代表一个训练样本。坐标轴 (x0,x1) 是输入，颜色代表样本所属的类别。训练完成后，模型将接收一个新样本 (x0,x1) 并预测其类别。  

虽然这个数据是生成的，但它代表了许多现实世界中的分类问题。其中有若干输入特征 (x0,...,xn) 和若干输出类别。模型经过训练，能够利用输入特征预测正确的输出类别。

In [4]:
# 显示数据集中的类别
print(f"unique classes {np.unique(y_train)}")
# 展示类别的表示方式
print(f"class representation {y_train[:10]}")
# 显示数据集的形状
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

unique classes [0 1 2 3]
class representation [3 3 3 0 3 3 3 3 2 0]
shape of X_train: (100, 2), shape of y_train: (100,)


## 2.2 模型
<img align="Right" src="./images/C2_W2_mclass_lab_network.PNG"  style=" width:350px; padding: 10px 20px ; ">
本实验将使用一个如图所示的 2 层网络。
与二分类网络不同，这个网络有四个输出，每个类别对应一个。给定一个输入样本，输出值最大的即为该输入的预测类别。   

下面是在 Tensorflow 中构建该网络的示例。请注意，输出层使用的是 `linear`（线性）激活而非 `softmax`。虽然可以在输出层中包含 softmax，但在训练时将线性输出传递给损失函数在数值上更稳定。如果使用模型预测概率，届时再应用 softmax 即可。

In [5]:
tf.random.set_seed(1234)  # applied to achieve consistent results
model = Sequential(
    [
        Dense(2, activation = 'relu',   name = "L1"),
        Dense(4, activation = 'linear', name = "L2")
    ]
)

下面的语句编译并训练网络。将损失函数的参数设为 `from_logits=True`，表明输出激活是线性的而非 softmax。

In [6]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

model.fit(
    X_train,y_train,
    epochs=200
)

Epoch 1/200
4/4 [==============================] - 0s 1ms/step - loss: 1.8158
Epoch 2/200
4/4 [==============================] - 0s 1ms/step - loss: 1.6976
Epoch 3/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5989
Epoch 4/200
4/4 [==============================] - 0s 961us/step - loss: 1.5179
Epoch 5/200
4/4 [==============================] - 0s 1ms/step - loss: 1.4369
Epoch 6/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3756
Epoch 7/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3154
Epoch 8/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2621
Epoch 9/200
4/4 [==============================] - 0s 991us/step - loss: 1.2188
Epoch 10/200
4/4 [==============================] - 0s 983us/step - loss: 1.1791
Epoch 11/200
4/4 [==============================] - 0s 974us/step - loss: 1.1446
Epoch 12/200
4/4 [==============================] - 0s 984us/step - loss: 1.1129
Epoch 13/200
4/4 [==============================] -

4/4 [==============================] - 0s 1ms/step - loss: 0.1758
Epoch 104/200
4/4 [==============================] - 0s 956us/step - loss: 0.1709
Epoch 105/200
4/4 [==============================] - 0s 977us/step - loss: 0.1662
Epoch 106/200
4/4 [==============================] - 0s 958us/step - loss: 0.1616
Epoch 107/200
4/4 [==============================] - 0s 952us/step - loss: 0.1575
Epoch 108/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1527
Epoch 109/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1480
Epoch 110/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1439
Epoch 111/200
4/4 [==============================] - 0s 996us/step - loss: 0.1396
Epoch 112/200
4/4 [==============================] - 0s 987us/step - loss: 0.1357
Epoch 113/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1315
Epoch 114/200
4/4 [==============================] - 0s 998us/step - loss: 0.1277
Epoch 115/200
4/4 [=====================

模型训练完成后，我们可以看到模型对训练数据的分类情况。

In [7]:
plt_cat_mc(X_train, y_train, model, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

上图中，决策边界展示了模型如何划分输入空间。这个非常简单的模型在对训练数据分类时毫无困难。它是如何做到的？让我们更详细地观察这个网络。

下面，我们将从模型中取出训练好的权重，并用它绘制网络中每个单元的函数。再往下还有对结果更详细的解释。要成功使用神经网络并不需要了解这些细节，但它可能有助于你更直观地理解各层如何协同解决分类问题。

In [8]:
# 从第一层收集训练好的参数
l1 = model.get_layer("L1")
W1,b1 = l1.get_weights()

In [9]:
# 绘制第一层的函数
plt_layer_relu(X_train, y_train.reshape(-1,), W1, b1, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [10]:
# 从输出层收集训练好的参数
l2 = model.get_layer("L2")
W2, b2 = l2.get_weights()
# 创建“新特征”，即经过 L1 变换后的训练样本
Xl2 = np.maximum(0, np.dot(X_train,W1) + b1)

plt_output_layer_linear(Xl2, y_train.reshape(-1,), W2, b2, classes,
                        x0_rng = (-0.25,np.amax(Xl2[:,0])), x1_rng = (-0.25,np.amax(Xl2[:,1])))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 解释
#### 第 1 层 <img align="Right" src="./images/C2_W2_mclass_layer1.png"  style=" width:600px; padding: 10px 20px ; ">
这些图展示了网络第一层中单元 0 和单元 1 的函数。坐标轴上的输入是 ($x_0,x_1$)。单元的输出由背景颜色表示，即每张图右侧的色条。请注意，由于这些单元使用 ReLu，输出不一定落在 0 和 1 之间，本例中峰值处大于 20。
图中的等高线显示了输出 $a^{[1]}_j$ 在零与非零之间的过渡点。回忆一下 ReLu 的图像：<img align="right" src="./images/C2_W2_mclass_relu.png"  style=" width:200px; padding: 10px 20px ; "> 图中的等高线就是 ReLu 的拐点。

单元 0 将类别 0 和 1 与类别 2 和 3 分开。线左侧的点（类别 0 和 1）输出为零，而线右侧的点输出大于零的值。  
单元 1 将类别 0 和 2 与类别 1 和 3 分开。线上方的点（类别 0 和 2）输出为零，而线下方的点输出大于零的值。让我们看看这在下一层中如何发挥作用！

#### 第 2 层，输出层  <img align="Right" src="./images/C2_W2_mclass_layer2.png"  style=" width:600px; padding: 10px 20px ; ">

这些图中的点是经过第一层变换后的训练样本。一种理解方式是：第一层为第二层的评估创建了一组新特征。这些图中的坐标轴是上一层的输出 $a^{[1]}_0$ 和 $a^{[1]}_1$。正如上面预测的，类别 0 和 1（蓝色和绿色）的 $a^{[1]}_0 = 0$，而类别 0 和 2（蓝色和橙色）的 $a^{[1]}_1 = 0$。  
同样，背景颜色的强度表示较高的值。  
单元 0 在接近 (0,0) 处产生最大值，类别 0（蓝色）就被映射到这里。    
单元 1 在左上角产生最高值，对应类别 1（绿色）。  
单元 2 对应右下角，类别 2（橙色）位于此处。  
单元 3 在右上角产生最高值，选中我们最后的类别（紫色）。  

从图中不太明显的另一点是，各单元之间的取值是经过协调的。一个单元仅仅为它要选择的类别产生最大值是不够的，它还必须在该类别的所有点上是所有单元中取值最高的。这是由损失函数（`SparseCategoricalCrossEntropy`）中隐含的 softmax 函数完成的。与其他激活函数不同，softmax 作用在所有输出之上。

你无需了解每个单元的细节就能成功使用神经网络。希望这个例子能让你对背后发生的事情有一些直观认识。

## 恭喜！
你已经学会了构建并运行一个用于多分类的神经网络。
